In [1]:
# All imports + the LLM client used in this notebook
from langchain_openai import ChatOpenAI
import os
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
# RunnableBranch = if / elif / else for chains
from langchain_core.runnables import RunnableSequence,RunnableParallel,RunnableLambda,RunnableBranch

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

In [2]:
# Quick demo: force the LLM to answer in a fixed structure instead of free text
from pydantic import BaseModel
from typing import Literal

# The schema: the LLM must return a field that is exactly "positive" or "Negative"
# (note the capital N - this matters later in the branch condition!)
class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive","Negative"]

# with_structured_output makes the LLM return a llm_schema object, not a string
llm_structured_output = llm.with_structured_output(llm_schema)

demo_chain = llm_structured_output

demo_chain.invoke("The movie was great")

llm_schema(movie_summary_flag='positive')

# Chapter 2.4 - Conditional chains

**Idea:** pick which chain to run based on a value computed earlier (like `if/else`).

1. LLM classifies a movie review as positive / negative (structured output).
2. `RunnableBranch` checks the label:
   - positive -> **LinkedIn** chain
   - otherwise -> **Instagram** chain

```
review -> prompt -> LLM(structured) -> label string -> RunnableBranch -+-> LinkedIn chain
                                                                       +-> Instagram chain
```

In [3]:
#Task-1[Prompt]
# Classification prompt: asks the LLM to label the review
prompt_template = ChatPromptTemplate.from_messages([
    ("system","you are a movie review evaluator"),
    ("user","Categorize the movie review as positive or negative: {input}")
])

In [4]:
#Task-2[LLM]
# LLM that returns an llm_schema object (label is guaranteed to be one of the 2 values)
llm_structured_output = llm.with_structured_output(llm_schema)

In [5]:
#task-3[custom runnable]
# Converts the pydantic object -> plain label string ("positive" / "Negative")
# so the branch condition can simply check the string.
def pydantic_json(input:llm_schema)->str:
    return input.model_dump()['movie_summary_flag']

pydantic_json_runnable = RunnableLambda(pydantic_json)

## Chain 1 - LinkedIn (used when review is positive)

In [6]:
#Task-1[Prompt]
linkedin_prompt =  ChatPromptTemplate.from_messages([
    ("system","You are a linkedin post generator"),
    ("human","Create a post for the following text for LinkedIn: {text}")
])

#Task-2[LLM]
llm = ChatOpenAI(base_url="https://integrate.api.nvidia.com/v1",model="nvidia/nemotron-3.5-lightning-30b-a3b")

#Task-3[Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser

## Chain 2 - Instagram (default branch, used otherwise)

In [9]:
def insta_chain(text:dict):
    # `text` is whatever reaches this branch (here: the label string, since
    # the branch is fed with the output of pydantic_json_runnable)

    #Task-1[Prompt]
    insta_prompt =  ChatPromptTemplate.from_messages([
        ("system","You are a Instagram post generator"),
        ("human","Create a post for the following text for Instagram: {text}")
    ])

    #Task-2[LLM]
    llm = ChatOpenAI(base_url="https://integrate.api.nvidia.com/v1",model="nvidia/nemotron-3.5-lightning-30b-a3b")

    #Task-3[Str Parser]
    str_parser = StrOutputParser()

    chain_instagram = insta_prompt | llm | str_parser
    result=chain_instagram.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)


## Final Orchestration
Connect the classifier to the branch.

> **Heads-up (known quirk):** the branch receives only the label (`"positive"` / `"Negative"`), not the
> original review, so the post is written about the *word* "Negative" rather than the review text.
> That is why the output below is generic. To fix it, pass the review along too
> (e.g. with `RunnablePassthrough` / `RunnableParallel`).
> Also the check `"positive" in x` is case-sensitive: `"Negative"` -> goes to the default (Instagram) branch.

In [10]:
# RunnableBranch((condition, chain), ..., default_chain)
# Conditions are checked in order; the first True one runs, else the last item (default).
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),   # if label contains "positive" -> LinkedIn
    insta_chain_runnable                           # else -> Instagram
)

# prompt -> structured LLM -> label string -> pick a chain based on the label
final_orchestrator = prompt_template | llm_structured_output | pydantic_json_runnable | conditional_chain

# Negative review -> label "Negative" -> Instagram branch
final_orchestrator.invoke({'input': 'I dont like this movie'})

'Here are 3 Instagram post options, each interpreting "Negative" differently so you can pick what fits your vibe/brand:\n\n---\n\n### 🖤 Option 1: Moody Aesthetic (Art/Visual-Focused)\n**Visual:** High-contrast black & white photo. Lots of negative space–maybe a single subject (a tree, a face, a coffee cup) against a solid dark or light background. Play with shadows and emptiness.\n\n**Caption:**\nthe space between the lines 🖤\nsometimes the most powerful thing you can leave out is everything else.\n(art by @yourhandle or original photo)\n\n**Hashtags:**\n#negative space #blackandwhitephotography #aesthetic Vibes #minimalism #visualart #ig_photo #artistic #tonalmood\n\n---\n\n### 💬 Option 2: Real & Relatable (Honest/Wellness-Focused)\n**Visual:** Candid, low-filter selfie or a cozy flat-lay (journal, tea, window light). Natural, unpolished, warm tones.\n\n**Caption:**\nhaving a "negative" day doesn\'t mean a "negative" life. \n✨ it\'s okay to feel what you\'re feeling without forcing po